# Description

Projects SMulTiXcan gene-trait associations and LINCS L1000 drug-gene expression profiles into the ARCHS4 CLAMP latent space.

The CLAMP projection uses the ridge-regularized formula:
$$B = (Z^T Z + \lambda I)^{-1} Z^T Y$$

where Z is the gene loading matrix from the CLAMP model, Y is the input data (genes × samples/traits/drugs), and λ is the L2 regularization parameter from the CLAMP model.

Genes missing from the input data are filled with 0 (mean). Gene symbols from CLAMP Z are mapped to Ensembl IDs via `clusterProfiler::bitr()` to align with the Ensembl-indexed LINCS and SMulTiXcan data.

Outputs saved to `output/drug_disease_analyses/`:
- `smultixcan-mashr-zscores-projection.pkl`: LVs × traits
- `lincs-projection.pkl`: LVs × drugs

# Module loading

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from pyprojroot import here

# Settings

In [23]:
# Data directory
DATA_DIR = here('data/archs4/drug_diseases_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

# CLAMP model
CLAMP_MODEL_FILE = here('output/archs4/archs4_CLAMP_C2CP.rds')
display(CLAMP_MODEL_FILE)
assert CLAMP_MODEL_FILE.exists()

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/archs4_CLAMP_C2CP.rds')

In [24]:
# Input data
LINCS_FILE = DATA_DIR / 'lincs-data.pkl'
display(LINCS_FILE)
assert LINCS_FILE.exists()

SMULTIXCAN_FILE = DATA_DIR / 'smultixcan-mashr-zscores.pkl'
display(SMULTIXCAN_FILE)
assert SMULTIXCAN_FILE.exists()

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations/lincs-data.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations/smultixcan-mashr-zscores.pkl')

In [25]:
# Output
OUTPUT_DIR = here('output/drug_disease_analyses')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses')

# Projection function

Implements the same formula as `CLAMP::projectCLAMP()` in Python/numpy:
```r
B <- solve(t(Z) %*% Z + L2 * diag(ncol(Z))) %*% (t(Z) %*% newdata)
```
Missing genes (not in input data) are filled with 0 (mean substitution).

In [26]:
def project_clamp(data_df, z_df, l2):
    """
    Projects data into CLAMP latent space using ridge-regularized formula:
        B = (Z^T Z + L2*I)^{-1} Z^T Y

    Args:
        data_df: DataFrame with genes (Ensembl IDs) in rows, samples/traits/drugs in columns.
        z_df: Z matrix DataFrame with Ensembl IDs in rows, LVs in columns.
        l2: L2 regularization scalar from CLAMP model.

    Returns:
        DataFrame with LVs in rows, samples/traits/drugs in columns.
    """
    # Align data to Z gene order; fill genes missing in data with 0
    y = data_df.reindex(z_df.index).fillna(0.0).values  # (n_genes, n_samples)
    Z = z_df.values  # (n_genes, n_lvs)
    K = Z.shape[1]

    ZtZ = Z.T @ Z
    B = np.linalg.solve(ZtZ + l2 * np.eye(K), Z.T @ y)

    return pd.DataFrame(B, index=z_df.columns, columns=data_df.columns)

# Load CLAMP model

In [27]:
readRDS = ro.r['readRDS']
clamp = readRDS(str(CLAMP_MODEL_FILE))
print('CLAMP model loaded')

CLAMP model loaded


In [28]:
# Extract Z matrix (gene loadings) and L2 regularization
Z_matrix = clamp.rx2('Z')
l2_value = float(clamp.rx2('L2')[0])

with localconverter(ro.default_converter + pandas2ri.converter):
    Z_values = ro.conversion.rpy2py(Z_matrix)

gene_symbols = list(ro.r['rownames'](Z_matrix))
lv_names = list(ro.r['colnames'](Z_matrix))

z_symbol = pd.DataFrame(
    data=Z_values,
    index=gene_symbols,
    columns=lv_names,
)

print(f'Z shape: {z_symbol.shape}')
print(f'L2: {l2_value}')
display(z_symbol.head())

Z shape: (18423, 2366)
L2: 410.97171081441064


,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,...,LV2357,LV2358,LV2359,LV2360,LV2361,LV2362,LV2363,LV2364,LV2365,LV2366
A1BG,0.000000,1.050998,0.000000,0.298539,0.0,0.571428,0.612714,0.59443,0.0,0.795284,...,0.0,0.0,0.534798,0.000000,0.000000,0.421127,0.000000,0.0,0.000000,0.0
A1BG-AS1,1.901109,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.00000,0.0,0.595865,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.457876,0.0
A2M,0.000000,0.471926,0.000000,0.000000,0.0,1.255996,0.000000,0.00000,0.0,0.000000,...,0.0,0.0,0.578630,0.380181,0.000000,0.413712,0.000000,0.0,0.000000,0.0
A2M-AS1,0.000000,0.919364,0.000000,0.000000,0.0,1.132544,0.000000,0.00000,0.0,0.000000,...,0.0,0.0,0.000000,1.117329,0.000000,1.618912,0.801983,0.0,0.650880,0.0
A2ML1,0.431602,1.413653,0.706858,0.000000,0.0,0.000000,0.000000,0.00000,0.0,0.000000,...,0.0,0.0,0.325118,0.000000,0.418311,0.000000,0.000000,0.0,0.000000,0.0


# Map CLAMP gene symbols → Ensembl IDs

LINCS and SMulTiXcan use Ensembl IDs; CLAMP Z uses HGNC gene symbols.
We use `clusterProfiler::bitr()` to map and keep only 1:1 unambiguous mappings.

In [29]:
clusterProfiler = importr('clusterProfiler')

bitr_result = clusterProfiler.bitr(
    ro.StrVector(gene_symbols),
    fromType='SYMBOL',
    toType='ENSEMBL',
    OrgDb='org.Hs.eg.db',
)

with localconverter(ro.default_converter + pandas2ri.converter):
    mapping_df = ro.conversion.rpy2py(bitr_result)

print(f'Raw mapping shape: {mapping_df.shape}')
display(mapping_df.head())

R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


Raw mapping shape: (19313, 2)


,SYMBOL,ENSEMBL
1,A1BG,ENSG00000121410
2,A1BG-AS1,ENSG00000268895
3,A2M,ENSG00000175899
4,A2M-AS1,ENSG00000245105
5,A2ML1,ENSG00000166535


In [30]:
# Keep only 1:1 mappings (drop symbols mapping to multiple Ensembl IDs and vice versa)
dup_symbols = mapping_df['SYMBOL'].duplicated(keep=False)
dup_ensembl = mapping_df['ENSEMBL'].duplicated(keep=False)
mapping_1to1 = mapping_df[~dup_symbols & ~dup_ensembl].set_index('SYMBOL')

print(f'1:1 mappings: {mapping_1to1.shape[0]} / {len(gene_symbols)} CLAMP genes')
display(mapping_1to1.head())

1:1 mappings: 16038 / 18423 CLAMP genes


,ENSEMBL
SYMBOL,
A1BG,ENSG00000121410
A1BG-AS1,ENSG00000268895
A2M,ENSG00000175899
A2M-AS1,ENSG00000245105
A2ML1,ENSG00000166535


In [31]:
# Rename Z index: gene symbol → Ensembl ID (keep only mapped genes)
z_ensembl = z_symbol.loc[mapping_1to1.index].rename(index=mapping_1to1['ENSEMBL'])

print(f'Z with Ensembl IDs shape: {z_ensembl.shape}')
display(z_ensembl.head())

Z with Ensembl IDs shape: (16038, 2366)


,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,...,LV2357,LV2358,LV2359,LV2360,LV2361,LV2362,LV2363,LV2364,LV2365,LV2366
SYMBOL,,,,,,,,,,,,,,,,,,,,,
ENSG00000121410,0.000000,1.050998,0.000000,0.298539,0.0,0.571428,0.612714,0.59443,0.0,0.795284,...,0.0,0.0,0.534798,0.000000,0.000000,0.421127,0.000000,0.0,0.000000,0.0
ENSG00000268895,1.901109,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.00000,0.0,0.595865,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.457876,0.0
ENSG00000175899,0.000000,0.471926,0.000000,0.000000,0.0,1.255996,0.000000,0.00000,0.0,0.000000,...,0.0,0.0,0.578630,0.380181,0.000000,0.413712,0.000000,0.0,0.000000,0.0
ENSG00000245105,0.000000,0.919364,0.000000,0.000000,0.0,1.132544,0.000000,0.00000,0.0,0.000000,...,0.0,0.0,0.000000,1.117329,0.000000,1.618912,0.801983,0.0,0.650880,0.0
ENSG00000166535,0.431602,1.413653,0.706858,0.000000,0.0,0.000000,0.000000,0.00000,0.0,0.000000,...,0.0,0.0,0.325118,0.000000,0.418311,0.000000,0.000000,0.0,0.000000,0.0


# Project SMulTiXcan data

In [32]:
smultixcan = pd.read_pickle(SMULTIXCAN_FILE)
print(f'SMulTiXcan shape: {smultixcan.shape}')
display(smultixcan.head())

SMulTiXcan shape: (22515, 4091)


,20096_1-Size_of_red_wine_glass_drunk_small_125ml,2345-Ever_had_bowel_cancer_screening,N49-Diagnoses_main_ICD10_N49_Inflammatory_disorders_of_male_genital_organs_not_elsewhere_classified,100011_raw-Iron,5221-Index_of_best_refractometry_result_right,20003_1141150624-Treatmentmedication_code_zomig_25mg_tablet,S69-Diagnoses_main_ICD10_S69_Other_and_unspecified_injuries_of_wrist_and_hand,20024_1136-Job_code_deduced_Information_and_communication_technology_managers,20002_1385-Noncancer_illness_code_selfreported_allergy_or_anaphylactic_reaction_to_food,G6_SLEEPAPNO-Sleep_apnoea,...,Astle_et_al_2016_Sum_basophil_neutrophil_counts,RA_OKADA_TRANS_ETHNIC,pgc.scz2,PGC_ADHD_EUR_2017,MAGIC_FastingGlucose,Astle_et_al_2016_Red_blood_cell_count,SSGAC_Depressive_Symptoms,BCAC_ER_positive_BreastCancer_EUR,IBD.EUR.Inflammatory_Bowel_Disease,Astle_et_al_2016_High_light_scatter_reticulocyte_count
gene_name,,,,,,,,,,,,,,,,,,,,,
ENSG00000000419,0.169468,0.102558,0.239545,0.887758,1.313448,1.472148,0.726160,1.516367,1.299771,1.068093,...,0.813014,0.275993,0.510834,0.024717,0.430951,0.824314,0.367414,1.377624,0.738444,0.298259
ENSG00000000457,1.358856,1.846875,0.139324,0.129530,0.757757,1.103979,0.612418,1.822327,2.035372,1.008058,...,1.441795,0.654791,2.545653,1.202984,0.514244,0.237223,0.414171,0.101731,1.012735,0.945167
ENSG00000000460,0.151008,1.173202,1.179426,0.571656,0.098771,0.221072,0.276415,0.461381,0.855502,0.201876,...,0.668962,0.300040,0.541782,1.033308,0.482261,0.695624,0.336480,0.083316,3.493196,0.991948
ENSG00000000938,1.302722,0.841524,1.578926,0.721340,0.139314,4.387016,0.125959,1.247123,0.215124,0.892083,...,0.126657,0.048048,1.886356,0.540496,0.127524,1.494501,0.056432,1.704863,1.351619,1.027297
ENSG00000000971,1.338813,0.262339,0.689379,1.702019,0.325859,0.063161,1.141126,0.882682,0.035533,1.810191,...,0.858497,1.675562,2.319072,1.598721,0.162958,0.005703,3.004544,0.803669,0.444266,0.165671


In [33]:
# Check gene ID format
print('First 5 SMulTiXcan gene IDs:', smultixcan.index[:5].tolist())
print('First 5 Z Ensembl IDs:', z_ensembl.index[:5].tolist())

# Count overlap
common = z_ensembl.index.intersection(smultixcan.index)
print(f'Common genes (CLAMP Z ∩ SMulTiXcan): {len(common)} / {len(z_ensembl.index)} CLAMP genes')

First 5 SMulTiXcan gene IDs: ['ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938', 'ENSG00000000971']
First 5 Z Ensembl IDs: ['ENSG00000121410', 'ENSG00000268895', 'ENSG00000175899', 'ENSG00000245105', 'ENSG00000166535']
Common genes (CLAMP Z ∩ SMulTiXcan): 13661 / 16038 CLAMP genes


In [34]:
# Drop traits with all-NaN gene associations; fill remaining NaNs with 0
n_before = smultixcan.shape[1]
smultixcan = smultixcan.dropna(axis=1, how='all').fillna(0.0)
print(f'Traits after dropping all-NaN columns: {smultixcan.shape[1]} (was {n_before})')

Traits after dropping all-NaN columns: 4091 (was 4091)


In [35]:
smultixcan_proj = project_clamp(smultixcan, z_ensembl, l2_value)
print(f'SMulTiXcan projection shape: {smultixcan_proj.shape}')
display(smultixcan_proj.head())

SMulTiXcan projection shape: (2366, 4091)


,20096_1-Size_of_red_wine_glass_drunk_small_125ml,2345-Ever_had_bowel_cancer_screening,N49-Diagnoses_main_ICD10_N49_Inflammatory_disorders_of_male_genital_organs_not_elsewhere_classified,100011_raw-Iron,5221-Index_of_best_refractometry_result_right,20003_1141150624-Treatmentmedication_code_zomig_25mg_tablet,S69-Diagnoses_main_ICD10_S69_Other_and_unspecified_injuries_of_wrist_and_hand,20024_1136-Job_code_deduced_Information_and_communication_technology_managers,20002_1385-Noncancer_illness_code_selfreported_allergy_or_anaphylactic_reaction_to_food,G6_SLEEPAPNO-Sleep_apnoea,...,Astle_et_al_2016_Sum_basophil_neutrophil_counts,RA_OKADA_TRANS_ETHNIC,pgc.scz2,PGC_ADHD_EUR_2017,MAGIC_FastingGlucose,Astle_et_al_2016_Red_blood_cell_count,SSGAC_Depressive_Symptoms,BCAC_ER_positive_BreastCancer_EUR,IBD.EUR.Inflammatory_Bowel_Disease,Astle_et_al_2016_High_light_scatter_reticulocyte_count
LV1,0.153456,0.171552,0.153334,0.152908,0.134566,0.163124,0.145335,0.146773,0.152961,0.142850,...,0.262381,0.196087,0.286988,0.163989,0.139514,0.285309,0.157414,0.156926,0.194079,0.297641
LV2,0.100974,0.106346,0.101445,0.095510,0.088895,0.089579,0.089118,0.090771,0.082718,0.102523,...,0.172808,0.141204,0.143710,0.121239,0.090788,0.199298,0.095521,0.101541,0.142240,0.164590
LV3,0.070661,0.083573,0.082994,0.065916,0.073920,0.086248,0.073543,0.076102,0.061243,0.069460,...,0.073540,0.090083,0.152892,0.113261,0.060385,0.090092,0.089826,0.072068,0.070305,0.086717
LV4,0.103140,0.111902,0.096378,0.085740,0.107958,0.111800,0.124310,0.087583,0.109811,0.118070,...,0.136074,0.113084,0.168764,0.136986,0.092374,0.133274,0.133228,0.112045,0.119645,0.094503
LV5,0.035550,0.015937,0.014084,-0.014893,0.015324,0.027896,0.024724,0.020440,0.018119,0.027276,...,0.042351,0.049341,0.024278,0.017188,0.025756,0.071969,0.003834,0.013244,0.049691,0.020399


In [36]:
output_file = OUTPUT_DIR / 'smultixcan-mashr-zscores-projection.pkl'
smultixcan_proj.to_pickle(output_file)
print(f'Saved to: {output_file}')

Saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/smultixcan-mashr-zscores-projection.pkl


# Project LINCS data

In [37]:
lincs = pd.read_pickle(LINCS_FILE)
print(f'LINCS shape: {lincs.shape}')
display(lincs.head())

LINCS shape: (7120, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
ENSG00000196839,-1.001,-1.835,1.391,1.132,0.257,1.932,0.508,1.408,0.777,0.032,...,-1.692,-0.516,-1.435,-0.317,-0.012,0.641,-0.230,-0.518,-0.177,2.146
ENSG00000170558,1.146,-1.863,0.011,-1.020,1.143,-0.115,1.327,0.310,-1.853,0.872,...,0.354,0.498,0.268,-1.084,-0.142,-0.077,0.633,-1.807,0.032,0.135
ENSG00000117020,-0.693,1.694,-0.804,-0.164,1.145,-1.465,1.221,-0.747,0.829,-0.961,...,-1.196,-0.230,-1.049,-0.347,0.586,0.865,-0.021,2.180,-0.956,0.105
ENSG00000133997,-0.037,0.383,0.269,-0.997,0.185,-0.536,0.424,-0.119,-1.313,0.579,...,-0.343,0.116,-0.245,-0.127,-1.367,0.149,0.117,2.084,1.178,0.772
ENSG00000101473,0.162,-0.899,0.105,-0.090,-1.291,1.404,0.185,0.157,-0.327,-0.026,...,-0.136,-1.115,-0.280,0.200,0.638,-0.197,-0.360,-2.302,-0.117,-0.167


In [38]:
# Check gene ID format and count overlap
print('First 5 LINCS gene IDs:', lincs.index[:5].tolist())

common_lincs = z_ensembl.index.intersection(lincs.index)
print(f'Common genes (CLAMP Z ∩ LINCS): {len(common_lincs)} / {len(z_ensembl.index)} CLAMP genes')

First 5 LINCS gene IDs: ['ENSG00000196839', 'ENSG00000170558', 'ENSG00000117020', 'ENSG00000133997', 'ENSG00000101473']
Common genes (CLAMP Z ∩ LINCS): 6590 / 16038 CLAMP genes


In [39]:
lincs_proj = project_clamp(lincs, z_ensembl, l2_value)
print(f'LINCS projection shape: {lincs_proj.shape}')
display(lincs_proj.head())

LINCS projection shape: (2366, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,-0.003664,0.021802,0.050683,-0.023411,0.054818,0.022504,-0.016074,-0.007949,-0.012020,-0.022674,...,-0.051500,-0.039418,-0.021987,0.012250,0.009535,0.012377,0.013959,-0.209899,0.026002,0.009327
LV2,0.011797,0.058622,0.001288,-0.023493,0.009715,0.001838,-0.025821,-0.026784,-0.010831,0.000237,...,-0.003300,-0.000286,0.004965,0.011235,0.002291,-0.003366,-0.002029,-0.067440,0.005340,0.011255
LV3,-0.000841,-0.055107,-0.006764,0.027218,-0.012003,-0.010472,-0.021370,-0.007238,-0.005116,0.005954,...,-0.003723,0.009973,-0.007469,0.009228,-0.010771,-0.017754,0.012783,0.066089,0.008973,-0.011303
LV4,0.031992,-0.511423,-0.071332,-0.059371,-0.035587,-0.033645,-0.076674,0.042806,-0.099315,0.031507,...,0.051060,0.022656,0.054687,0.014682,-0.078138,-0.014696,0.015666,-0.238956,-0.018649,-0.047013
LV5,-0.032510,0.123403,0.005948,0.031886,-0.045581,-0.006246,0.092339,0.034634,0.111000,-0.025291,...,-0.020998,-0.019916,-0.030282,0.009627,0.031024,-0.017982,-0.019684,0.164316,0.012215,-0.027974


In [40]:
output_file = OUTPUT_DIR / 'lincs-projection.pkl'
lincs_proj.to_pickle(output_file)
print(f'Saved to: {output_file}')

Saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs-projection.pkl
